# 03 · Recuperación semántica de reseñas
Idea: convertir cada reseña en un **embedding** (vector de 384 dimensiones) con `sentence-transformers/all-MiniLM-L6-v2`. Reseñas con significado parecido quedan cerca, aunque no compartan palabras ("battery dies fast" ≈ "doesn't hold a charge").

Búsqueda = similitud coseno entre el vector de la consulta y los de las reseñas → top-k. Se puede filtrar por sentimiento (etiqueta real o predicha por el modelo del notebook 02).

En Colab: **Entorno de ejecución → Cambiar tipo → GPU T4** (codifica 50k reseñas en ~1–2 min; en CPU ~10–15 min).

In [ ]:
!pip -q install sentence-transformers pandas pyarrow scikit-learn

In [ ]:
import os, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
SEED = 42

In [ ]:
import os, glob
RUTA = "data/electronics_50k.parquet"
if not os.path.exists(RUTA):
    try:                                   # en Colab: sube el parquet generado por el notebook 01
        from google.colab import files
        os.makedirs("data", exist_ok=True)
        up = files.upload()
        os.replace(list(up)[0], RUTA)
    except ImportError:
        RUTA = "../data/electronics_50k.parquet"   # ejecución local desde notebooks/
df = pd.read_parquet(RUTA)
df["label"] = df["label"].astype(str)
print(df.shape); df["label"].value_counts()

## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
modelo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
textos = df["review"].str.slice(0, 1000).tolist()        # MiniLM corta a 256 tokens
EMB = "data/embeddings_minilm.npy"
if os.path.exists(EMB):
    E = np.load(EMB)
else:
    E = modelo.encode(textos, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
    np.save(EMB, E)
E.shape

## Buscador
Con vectores normalizados, el coseno es un producto punto. Para 50k reseñas numpy basta; con millones se usaría FAISS.

In [ ]:
def buscar(consulta, k=5, sentimiento=None):
    q = modelo.encode([consulta], normalize_embeddings=True)[0]
    sims = E @ q
    if sentimiento:
        sims = np.where(df["label"].values == sentimiento, sims, -1)
    idx = np.argsort(-sims)[:k]
    out = df.iloc[idx][["review", "label", "rating"]].copy()
    out.insert(0, "sim", sims[idx].round(3))
    out["review"] = out["review"].str.slice(0, 160)
    return out.reset_index(drop=True)

pd.set_option("display.max_colwidth", 160)
buscar("battery dies fast")

In [ ]:
buscar("great sound quality for the price", sentimiento="positivo")

In [ ]:
buscar("stopped working after a month", sentimiento="negativo")

## Comparación con búsqueda por palabras clave (TF-IDF)
Para mostrar en la defensa qué aporta lo semántico: la misma consulta con TF-IDF solo encuentra coincidencias literales.

In [ ]:
tf = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True).fit(textos)
T = tf.transform(textos)
def buscar_tfidf(consulta, k=5):
    s = (T @ tf.transform([consulta]).T).toarray().ravel()
    idx = np.argsort(-s)[:k]
    return pd.DataFrame({"sim": s[idx].round(3), "review": df["review"].iloc[idx].str.slice(0,160).values})
buscar_tfidf("doesn't hold a charge")

In [ ]:
buscar("doesn't hold a charge")

## Evaluación: precision@5 con juicio manual
Para cada consulta, mira las 5 reseñas y marca 1 si es relevante y 0 si no. Se hace para ambos buscadores.

In [ ]:
CONSULTAS = [
    "battery dies fast", "doesn't hold a charge", "great sound quality for the price",
    "stopped working after a month", "easy to set up", "bluetooth keeps disconnecting",
    "screen is too dim", "cable broke quickly", "fast shipping well packaged",
    "overheats when charging", "comfortable headphones for long use", "remote control does not work",
]
filas = []
for q in CONSULTAS:
    for metodo, f in [("semantica", buscar), ("tfidf", buscar_tfidf)]:
        for rank, r in f(q, 5).iterrows():
            filas.append({"consulta": q, "metodo": metodo, "rank": rank+1, "review": r["review"], "relevante": None})
juicio = pd.DataFrame(filas)
juicio.to_csv("data/juicio_p5.csv", index=False)
print("Abre data/juicio_p5.csv, llena la columna 'relevante' con 1/0, súbelo y corre la celda siguiente.")

In [ ]:
j = pd.read_csv("data/juicio_p5.csv")
if j["relevante"].notna().all():
    p5 = j.groupby(["metodo", "consulta"])["relevante"].mean().unstack(0)
    print(p5.round(2)); print("\nPrecision@5 promedio:\n", p5.mean().round(3))
else:
    print("Faltan juicios por llenar.")

## Notas para la defensa
- **Semántica vs palabras clave:** la búsqueda semántica encuentra paráfrasis ("doesn't hold a charge" ↔ "battery drains overnight"); TF-IDF no.
- **Combinación con la parte A:** filtrar por sentimiento permite preguntas como "¿qué quejas hay sobre la batería?".
- **Limitaciones:** MiniLM está entrenado en inglés; la evaluación es manual y pequeña (12 consultas).